# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AxelYoel/FlyRank-AI-Internship---Axel-Yoel-Chandra/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

**Finding #8: Refreshed old content matches young fresh content**

- **Where the label comes from:** the Health Score is a FlyRank composite (impressions 30 pts, position 30, CTR 20, scroll depth 20). The claim compares the 365+ day old, updated 0-30 days cell (44.62) with the 31-90 day old, updated 0-30 days cell (44.12).
- **What would make it mislead:** the paper acknowledges survivor bias only for the small 365+ x 361+ cell. I would ask whether the active-content filter (impressions and sessions above zero) also affects the refreshed cell. There may also be selection bias if high-potential old pages were refreshed first.
- **My methodology question:** Finding #4 reports old pages refreshed within 30 days going from health 10.7 to 34.5, while the Finding #8 heatmap lists the 365+ x 0-30 cell at 44.62. Could the authors clarify whether "365+" means content age or days since update, and how the cohorts differ?

**Finding #3: Click capture falls sharply by position tier**

- **Where the label comes from:** weighted CTR is total clicks divided by total impressions in each tier (0.423% for Top 3 down to 0.050% for Deep). The 88% drop follows from those two numbers.
- **What would make it mislead:** the mix of query intent and SERP features may differ across tiers, and a page's average position blends many queries, so a tier describes a page rather than each query.
- **My methodology questions:** (1) How were rows with avg_position = 0 handled? (2) Are tiers built from individual queries or from page-level average positions?

In [10]:
# Section 1: check the paper's arithmetic (values copied from the paper)
top3, deep = 0.423, 0.050
print(f"Drop from Top 3 to Deep: {(top3 - deep) / top3:.1%} (paper says 88%)")

Drop from Top 3 to Deep: 88.2% (paper says 88%)


## 2. My model under an honest split (before/after)

**Prediction (written before running):** I expected the random split to score higher.

**Result** (same model and features, only the split changes):

| | random split | grouped by client |
|---|---|---|
| R2 | 0.616 | 0.613 |
| precision@20 | 0.41 | 0.41 |
| share of lost clicks in top 100 | 0.227 | 0.213 |

R² and precision@20 are essentially equal (0.616 vs 0.613, and 0.41 vs 0.41). Share of lost clicks in the top 100 is slightly higher under random (0.227 vs 0.213). Before I sorted the rows, two earlier runs gave random R² of 0.608 and 0.622, so R² differences of about 0.01 are within run-to-run noise. Share100 was higher under random in all three runs, a small consistent edge much smaller than the spread between folds (grouped R² ranges from about 0.36 to 0.87).

**Best explanation (not proven):** the model mostly relies on each page's own February traffic, which works the same way for every client, and client identity is not a feature, so there is little client-specific information to memorize.

**What this does not show:** it does not show the model is leak-free. A grouped split only protects against the same client appearing in training and testing. It uses one seed and one month.

In [11]:
%pip -q install duckdb
import duckdb, os
from google.colab import userdata

hf_token = userdata.get("HF_token")  # same secret name you used in W05
os.environ["HF_token"] = hf_token

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{hf_token}')")
rel = "hf://datasets/FlyRank/internship-warehouse"

model_data = con.sql(f"""
    WITH daily AS (
        SELECT content_hash_id, gsc_avg_position, gsc_impressions, gsc_clicks
        FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/data_0.parquet')
        WHERE gsc_data_available = TRUE AND gsc_avg_position > 0 AND gsc_impressions > 0
    ),
    rollup AS (
        SELECT
            content_hash_id,
            SUM(gsc_impressions) AS impressions,
            SUM(gsc_clicks) AS clicks,
            SUM(gsc_avg_position * gsc_impressions) / SUM(gsc_impressions) AS weighted_avg_position
        FROM daily
        GROUP BY content_hash_id
        HAVING SUM(gsc_impressions) >= 50
    ),
    tiered AS (
        SELECT *,
            CASE
                WHEN weighted_avg_position <= 3 THEN 'tier_1_1-3'
                WHEN weighted_avg_position <= 10 THEN 'tier_2_4-10'
                WHEN weighted_avg_position <= 20 THEN 'tier_3_11-20'
                ELSE 'tier_4_21plus'
            END AS position_tier
        FROM rollup
    ),
    benchmark(position_tier, expected_ctr) AS (
        VALUES
            ('tier_1_1-3', 0.002811), ('tier_2_4-10', 0.002353),
            ('tier_3_11-20', 0.002226), ('tier_4_21plus', 0.000748)
    ),
    labeled AS (
        SELECT
            t.content_hash_id, t.impressions, t.clicks, t.weighted_avg_position,
            b.expected_ctr,
            (b.expected_ctr * t.impressions) - t.clicks AS lost_clicks
        FROM tiered t
        JOIN benchmark b USING (position_tier)
    )
    SELECT
        l.content_hash_id, c.client_hash_id,
        l.lost_clicks, l.impressions, l.clicks, l.weighted_avg_position, l.expected_ctr,
        c.search_volume, c.competition_level, c.content_type,
        CASE WHEN c.content_updated_date <= DATE '2026-03-31'
             THEN DATE_DIFF('day', c.content_updated_date, DATE '2026-03-31')
             ELSE NULL END AS days_since_update_capped,
        c.word_count
    FROM labeled l
    JOIN read_parquet('{rel}/dim_content.parquet') c
        ON l.content_hash_id = c.content_hash_id
    WHERE c.content_type = 'keyword article' AND c.search_volume >= 20
""").df().sort_values('content_hash_id').reset_index(drop=True)

print(f"Rows: {len(model_data)}, distinct clients: {model_data['client_hash_id'].nunique()}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 38726, distinct clients: 43


In [12]:
import numpy as np
import pandas as pd

# page-detail features (same as W05)
model_data['has_word_count'] = model_data['word_count'].notna().astype(int)
model_data['word_count'] = model_data['word_count'].fillna(0)
model_data['updated_before_march31'] = model_data['days_since_update_capped'].notna().astype(int)
model_data['days_since_update_capped'] = model_data['days_since_update_capped'].fillna(0)
model_data = pd.get_dummies(model_data, columns=['competition_level'], drop_first=True)
model_data['capture_ratio'] = model_data['clicks'] / model_data['search_volume']

honest_features = ['search_volume', 'word_count', 'has_word_count',
                   'days_since_update_capped', 'updated_before_march31'] + \
                  [c for c in model_data.columns if c.startswith('competition_level_')]

# February traffic (same as W05)
feb = con.sql(f"""
    SELECT content_hash_id,
           SUM(gsc_impressions) AS feb_impressions,
           SUM(gsc_clicks) AS feb_clicks
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-02/data_0.parquet')
    WHERE gsc_data_available = TRUE
    GROUP BY content_hash_id
""").df()

model_data = model_data.merge(feb, on='content_hash_id', how='left')
model_data['has_feb_data'] = model_data['feb_impressions'].notna().astype(int)
model_data[['feb_impressions', 'feb_clicks']] = model_data[['feb_impressions', 'feb_clicks']].fillna(0)

feb_features = honest_features + ['feb_impressions', 'feb_clicks', 'has_feb_data']

print("Features:", feb_features)
print("Rows:", len(model_data), "| clients:", model_data['client_hash_id'].nunique(),
      "| share with Feb data:", model_data['has_feb_data'].mean().round(3))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Features: ['search_volume', 'word_count', 'has_word_count', 'days_since_update_capped', 'updated_before_march31', 'competition_level_LOW', 'competition_level_MEDIUM', 'feb_impressions', 'feb_clicks', 'has_feb_data']
Rows: 38726 | clients: 43 | share with Feb data: 0.877


In [13]:
from sklearn.model_selection import KFold, GroupKFold
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

def precision_at_k(true_ids, pred_ids, k=20):
    return len(set(true_ids[:k]) & set(pred_ids[:k])) / k

def top_k_share(test_df, ids, k=100):
    pos = test_df['lost_clicks'].clip(lower=0)
    return pos[test_df['content_hash_id'].isin(ids[:k]).values].sum() / pos.sum()

def run_split(data, feats, splits):
    rows = []
    for i, (tr, te) in enumerate(splits):
        tr_df, te_df = data.iloc[tr], data.iloc[te].copy()
        lin = LinearRegression().fit(tr_df[feats], tr_df['lost_clicks'])
        te_df['pred'] = lin.predict(te_df[feats])
        true_ids = te_df.sort_values('lost_clicks', ascending=False)['content_hash_id'].tolist()
        pred_ids = te_df.sort_values('pred', ascending=False)['content_hash_id'].tolist()
        rows.append({'fold': i, 'n_test': len(te_df),
                     'r2': r2_score(te_df['lost_clicks'], te_df['pred']),
                     'p20': precision_at_k(true_ids, pred_ids),
                     'chance_p20': 20 / len(te_df),
                     'share100': top_k_share(te_df, pred_ids)})
    return pd.DataFrame(rows)

X, y, g = model_data[feb_features], model_data['lost_clicks'], model_data['client_hash_id']
random_res  = run_split(model_data, feb_features, KFold(5, shuffle=True, random_state=42).split(X))
grouped_res = run_split(model_data, feb_features, GroupKFold(5).split(X, y, groups=g))

print(random_res.round(3)); print(grouped_res.round(3))
print(pd.DataFrame({'random': random_res.mean(), 'grouped': grouped_res.mean()}).round(3))

   fold  n_test     r2   p20  chance_p20  share100
0     0    7746  0.652  0.40       0.003     0.230
1     1    7745  0.855  0.55       0.003     0.207
2     2    7745  0.377  0.40       0.003     0.221
3     3    7745  0.664  0.30       0.003     0.253
4     4    7745  0.534  0.40       0.003     0.223
   fold  n_test     r2   p20  chance_p20  share100
0     0   11127  0.739  0.45       0.002     0.181
1     1    7163  0.652  0.40       0.003     0.217
2     2    6812  0.358  0.35       0.003     0.219
3     3    6812  0.874  0.60       0.003     0.241
4     4    6812  0.444  0.25       0.003     0.205
              random   grouped
fold           2.000     2.000
n_test      7745.200  7745.200
r2             0.616     0.613
p20            0.410     0.410
chance_p20     0.003     0.003
share100       0.227     0.213


## 3. Leakage audit

**Deliberate leak test:** I added expected_clicks (expected_ctr x impressions) and March clicks as features. R2 and precision@20 reached 1.0 in every fold, so my test setup can detect an obvious leak. Share of lost clicks in the top 100 only reached 0.339, which is the ceiling for that metric. My honest score (0.213) is about 63% of that ceiling.

**Timeline check:** days_since_update_capped was measured up to 2026-03-31, inside the month I am predicting. I rebuilt it as of 2026-02-28. 63 of 5,132 pages changed, and the scores were identical, so this slip was not driving the result. The fixed version stays in the final features.

**Population disclosure:** pages are included if they have 50 or more impressions in March. A real user on Feb 28 could not know which pages will reach that. 3,125 of 32,537 February-active pages (9.6%) never entered the March table, mostly low-traffic ones. I do not know whether this filter makes my scores look better or worse. My scores describe how well the model ranks keyword articles with search volume of 20 or more and at least 50 impressions in March, and they may not hold for other pages or content types.

**Benchmark check:** the expected-CTR rates used to build the label were computed from March data across all clients, including clients I hide in testing. Rebuilding them from training clients only changed R2 from 0.613 to 0.603, precision@20 from 0.41 to 0.40, and share100 from 0.213 to 0.212. The two giant-client folds moved the most. This is a small leak in one month of data.

**Other disclosures:** search_volume and competition_level come from a snapshot table with no date of its own. The W04 rule uses March clicks, so it is not a like-for-like baseline for a forecast. June 2026 was not used to build the label, choose features, set the filter, or compare models. No June data was queried in W01 to W05, so no June results exist yet.

In [14]:
# Deliberate leak: build expected_clicks from March columns, add it plus March clicks
model_data['expected_clicks'] = model_data['expected_ctr'] * model_data['impressions']
leaky_features = feb_features + ['expected_clicks', 'clicks']

leak_res = run_split(model_data, leaky_features, GroupKFold(5).split(X, y, groups=g))
print(leak_res.round(3))
print(pd.DataFrame({'honest (grouped)': grouped_res[['r2','p20','share100']].mean(),
                    'with leak (grouped)': leak_res[['r2','p20','share100']].mean()}).round(3))

   fold  n_test   r2  p20  chance_p20  share100
0     0   11127  1.0  1.0       0.002     0.245
1     1    7163  1.0  1.0       0.003     0.290
2     2    6812  1.0  1.0       0.003     0.389
3     3    6812  1.0  1.0       0.003     0.381
4     4    6812  1.0  1.0       0.003     0.391
          honest (grouped)  with leak (grouped)
r2                   0.613                1.000
p20                  0.410                1.000
share100             0.213                0.339


In [15]:
# 1. get the raw update date back
upd = con.sql(f"""
    SELECT content_hash_id, content_updated_date
    FROM read_parquet('{rel}/dim_content.parquet')
""").df()
n_before = len(model_data)
model_data = model_data.merge(upd, on='content_hash_id', how='left')
assert len(model_data) == n_before, "merge changed the row count"

# 2. rebuild the feature as of Feb 28
cutoff = pd.Timestamp('2026-02-28')
d = pd.to_datetime(model_data['content_updated_date'])
known = (d <= cutoff).values
model_data['days_since_update_feb'] = np.where(known, (cutoff - d).dt.days, 0)
model_data['updated_before_feb28'] = known.astype(int)

# 3. how many pages change?
changed = (model_data['updated_before_march31'] == 1) & (~known)
print("Pages whose 'known update' was really after Feb 28:", changed.sum(),
      "of", (model_data['updated_before_march31'] == 1).sum())

# 4. re-score with the fixed feature
fixed_features = [f for f in feb_features
                  if f not in ['days_since_update_capped', 'updated_before_march31']] \
                 + ['days_since_update_feb', 'updated_before_feb28']
X, y, g = model_data[fixed_features], model_data['lost_clicks'], model_data['client_hash_id']
fixed_res = run_split(model_data, fixed_features, GroupKFold(5).split(X, y, groups=g))
print(pd.DataFrame({'before fix': grouped_res[['r2','p20','share100']].mean(),
                    'after fix': fixed_res[['r2','p20','share100']].mean()}).round(3))

Pages whose 'known update' was really after Feb 28: 63 of 5132
          before fix  after fix
r2             0.613      0.613
p20            0.410      0.410
share100       0.213      0.213


In [16]:
feb_pop = con.sql(f"""
    WITH daily AS (
        SELECT content_hash_id, gsc_avg_position, gsc_impressions, gsc_clicks
        FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-02/data_0.parquet')
        WHERE gsc_data_available = TRUE AND gsc_avg_position > 0 AND gsc_impressions > 0
    ),
    rollup AS (
        SELECT content_hash_id,
               SUM(gsc_impressions) AS feb_impr_f,
               SUM(gsc_clicks) AS feb_clicks_f
        FROM daily
        GROUP BY content_hash_id
        HAVING SUM(gsc_impressions) >= 50
    )
    SELECT r.*, c.content_type, c.search_volume
    FROM rollup r
    JOIN read_parquet('{rel}/dim_content.parquet') c USING (content_hash_id)
    WHERE c.content_type = 'keyword article' AND c.search_volume >= 20
""").df()

feb_pop['in_march_table'] = feb_pop['content_hash_id'].isin(model_data['content_hash_id'])

n = len(feb_pop)
kept = int(feb_pop['in_march_table'].sum())
print(f"Feb-active pages: {n} | in March table: {kept} | dropped: {n - kept} "
      f"({(n - kept) / n:.1%})")

# does the drop depend on how busy the page was in February?
feb_pop['feb_bucket'] = pd.cut(feb_pop['feb_impr_f'], [49, 100, 500, 2000, float('inf')])
print(feb_pop.groupby('feb_bucket', observed=True)['in_march_table']
      .agg(['mean', 'size']).rename(columns={'mean': 'share_kept', 'size': 'n_pages'}).round(3))

# the other direction: March-table pages that were NOT in this February list
print("March-table pages not in the Feb list:",
      int((~model_data['content_hash_id'].isin(feb_pop['content_hash_id'])).sum()))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feb-active pages: 32537 | in March table: 29412 | dropped: 3125 (9.6%)
                 share_kept  n_pages
feb_bucket                          
(49.0, 100.0]         0.740     4353
(100.0, 500.0]        0.871    11532
(500.0, 2000.0]       0.954     9052
(2000.0, inf]         0.989     7600
March-table pages not in the Feb list: 9314


In [17]:
# 1. All March pages with 50+ impressions (the population your W03 benchmark used), with client id
allp = con.sql(f"""
    WITH daily AS (
        SELECT content_hash_id, gsc_avg_position, gsc_impressions, gsc_clicks
        FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/data_0.parquet')
        WHERE gsc_data_available = TRUE AND gsc_avg_position > 0 AND gsc_impressions > 0
    ),
    rollup AS (
        SELECT content_hash_id,
               SUM(gsc_impressions) AS impressions,
               SUM(gsc_clicks) AS clicks,
               SUM(gsc_avg_position * gsc_impressions) / SUM(gsc_impressions) AS weighted_avg_position
        FROM daily GROUP BY content_hash_id
        HAVING SUM(gsc_impressions) >= 50
    )
    SELECT r.*, c.client_hash_id
    FROM rollup r
    LEFT JOIN read_parquet('{rel}/dim_content.parquet') c USING (content_hash_id)
""").df()
print("pages:", len(allp), "(W03 had 114,447)")

def tier(pos):
    return np.select([pos <= 3, pos <= 10, pos <= 20], [0, 1, 2], default=3)

allp['tier'] = tier(allp['weighted_avg_position'].values)
allp['ctr'] = allp['clicks'] / allp['impressions']

def wmedian_ctr(df):
    out = {}
    for t, gdf in df.groupby('tier'):
        gdf = gdf.sort_values('ctr')
        cum = gdf['impressions'].cumsum().values
        out[t] = gdf['ctr'].values[np.searchsorted(cum, 0.5 * gdf['impressions'].sum())]
    return out

# SAFETY CHECK: should be close to 0.002811, 0.002353, 0.002226, 0.000748
print({k: round(v, 6) for k, v in wmedian_ctr(allp).items()})

# 2. Fold-by-fold: benchmark from training clients only
model_data['tier'] = tier(model_data['weighted_avg_position'].values)

def run_split_bench(data, feats, splits):
    rows = []
    for i, (tr, te) in enumerate(splits):
        test_clients = set(data.iloc[te]['client_hash_id'])
        bench = wmedian_ctr(allp[~allp['client_hash_id'].isin(test_clients)])
        d = data.copy()
        d['lost_clicks'] = d['tier'].map(bench).values * d['impressions'].values - d['clicks'].values
        tr_df, te_df = d.iloc[tr], d.iloc[te].copy()
        lin = LinearRegression().fit(tr_df[feats], tr_df['lost_clicks'])
        te_df['pred'] = lin.predict(te_df[feats])
        true_ids = te_df.sort_values('lost_clicks', ascending=False)['content_hash_id'].tolist()
        pred_ids = te_df.sort_values('pred', ascending=False)['content_hash_id'].tolist()
        rows.append({'fold': i, 'n_test': len(te_df),
                     'bench_top3': round(bench[0], 6), 'bench_deep': round(bench[3], 6),
                     'r2': r2_score(te_df['lost_clicks'], te_df['pred']),
                     'p20': precision_at_k(true_ids, pred_ids),
                     'share100': top_k_share(te_df, pred_ids)})
    return pd.DataFrame(rows)

X, y, g = model_data[fixed_features], model_data['lost_clicks'], model_data['client_hash_id']
bench_res = run_split_bench(model_data, fixed_features, GroupKFold(5).split(X, y, groups=g))
print(bench_res.round(3))
print(pd.DataFrame({'benchmark from all clients': fixed_res[['r2', 'p20', 'share100']].mean(),
                    'benchmark from training clients only': bench_res[['r2', 'p20', 'share100']].mean()}).round(3))

pages: 114447 (W03 had 114,447)
{0: np.float64(0.002811), 1: np.float64(0.002353), 2: np.float64(0.002226), 3: np.float64(0.000748)}
   fold  n_test  bench_top3  bench_deep     r2   p20  share100
0     0   11127       0.003       0.001  0.716  0.40     0.178
1     1    7163       0.003       0.001  0.624  0.40     0.214
2     2    6812       0.003       0.001  0.360  0.35     0.229
3     3    6812       0.003       0.001  0.875  0.60     0.240
4     4    6812       0.003       0.001  0.441  0.25     0.201
          benchmark from all clients  benchmark from training clients only
r2                             0.613                                 0.603
p20                            0.410                                 0.400
share100                       0.213                                 0.212


## 4. Claim rewrite

**Methods:** the W04 rule uses March clicks, so it is not a true forecast. The carry-forward rule uses February traffic plus March-derived benchmark rates. The model and the "February impressions only" rule use information known before March.

**Observed:** I examined 3 of 20 misses and 3 of 11 false alarms. The three misses had March impressions about 5x to 49x higher than February. Of the three false alarms, one had a sharp drop (51,000 February impressions to 2,288 in March); in the other two, impressions were similar in both months and March clicks were higher than February.

**Guess:** I did not check what caused these changes. Possible explanations include a new backlink or a seasonal change, but I have not tested any of them.

**Use:** use this as decision support only, to help prioritize which pages to review. It rests on one month and six examined pages, and it is not a rule or proof.

In [18]:
# Section 4: values copied from my W05 output, not recomputed here
# (March impressions, Feb impressions) for the three misses
for name, (mar, feb) in {'miss_1': (143019, 28998), 'miss_2': (71513, 1467),
                         'miss_3': (89229, 17891)}.items():
    print(name, f"March/Feb impressions = {mar / feb:.1f}x")

# (March impr, Feb impr, March clicks, Feb clicks) for the three false alarms
for name, (mi, fi, mc, fc) in {'fa_1': (2288, 51000, 1, 13), 'fa_2': (91388, 79986, 85, 74),
                               'fa_3': (59109, 55920, 150, 79)}.items():
    print(name, f"impressions {mi / fi:.2f}x, clicks {mc / fc:.2f}x")

miss_1 March/Feb impressions = 4.9x
miss_2 March/Feb impressions = 48.7x
miss_3 March/Feb impressions = 5.0x
fa_1 impressions 0.04x, clicks 0.08x
fa_2 impressions 1.14x, clicks 1.15x
fa_3 impressions 1.06x, clicks 1.90x


## Self-check

Before you submit, confirm each line honestly:

- [Done] Every section above is filled — markdown thinking AND the code that backs it
- [Done] The notebook runs top to bottom with no errors (Runtime → Run all)
- [Done] No client names, URLs, or private queries anywhere
- [Done] My claims use careful words: observed, measured, directional, decision-support
- [Done] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.